In [ ]:
# Ejemplo numérico: SGD vs Momentum vs Adam (toy)
import numpy as np

def g(theta):
    return 2*theta

theta0 = 1.0

# SGD
eta_sgd = 0.1
theta_sgd = theta0 - eta_sgd * g(theta0)

# Momentum (dos pasos)
gamma = 0.9
v0 = 0.0
v1 = gamma*v0 + eta_sgd * g(theta0)
theta_mom1 = theta0 - v1
v2 = gamma*v1 + eta_sgd * g(theta_mom1)
theta_mom2 = theta_mom1 - v2

# Adam (primer paso aproximado)
eta_adam = 0.01
beta1 = 0.9
beta2 = 0.999
eps = 1e-8
m1 = (1-beta1)*g(theta0)
v1_adam = (1-beta2)*(g(theta0)**2)
# correcciones de sesgo (primer paso)
mhat = m1/(1-beta1)
vhat = v1_adam/(1-beta2)
delta = eta_adam * mhat / (np.sqrt(vhat) + eps)
theta_adam = theta0 - delta

print('theta0 =', theta0)
print('SGD step ->', theta_sgd)
print('Momentum step1 ->', theta_mom1, ' step2 ->', theta_mom2)
print('Adam approx step ->', theta_adam)


### Ejemplo numérico rápido

Pequeño ejemplo ilustrativo: actualización de un parámetro para una función simple $L(\theta)=\theta^2$. Compara un paso de SGD, dos pasos con `momentum`, y la primera actualización aproximada de Adam. El siguiente bloque de código muestra los valores numéricos.


In [ ]:
# Celda opcional: instalar dependencias si faltan (útil fuera de Colab)
import importlib, sys, subprocess
pkgs = []
for p in ("tensorflow","numpy","matplotlib","scikit-learn"):
    if importlib.util.find_spec(p) is None:
        pkgs.append(p)
if pkgs:
    print('Instalando paquetes:', pkgs)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
else:
    print('Todos los paquetes principales ya están instalados.')

# Montar Google Drive (descomentar en Colab si quieres persistencia):
# from google.colab import drive
# drive.mount('/content/drive')


## Colab — instrucciones rápidas

- Para ejecutar en Google Colab: sube este notebook en https://colab.research.google.com/ (Upload) y activa Runtime → Change runtime type → GPU.
- Si quieres persistir resultados monta Google Drive (celda de ejemplo más abajo).
- Si faltan paquetes, ejecuta la celda de instalación que aparece a continuación.


## Requisitos y ejecución

- Requisitos recomendados (local): Python 3.8+ y los siguientes paquetes:
  - tensorflow, numpy, matplotlib

- Comandos sugeridos (conda/mamba):
```bash
mamba create -n ud4-optimizers -y python=3.10
mamba activate ud4-optimizers
mamba install -y -c conda-forge tensorflow matplotlib numpy
```

- En Google Colab no necesitas instalar nada; para persistencia monta Drive si lo deseas:
  - Runtime → Change runtime type → GPU para activar GPU.

- Para abrir rápido en Colab: visita https://colab.research.google.com/ y sube este notebook, o instala la extensión "Open in Colab" en VS Code.


# Práctico: Optimizadores en Keras

[Open in Colab](https://colab.research.google.com/) — sube este notebook a Colab o ábrelo desde Drive.

Este notebook muestra ejemplos prácticos de uso de optimizadores en `tf.keras` (SGD, SGD+momentum, RMSProp, Adam, AdamW).
Objetivo: comparar convergencia y comportamiento en un problema pequeño (Fashion‑MNIST).

## Instrucciones
- Ejecuta las celdas secuencialmente.
- Ajusta `EPOCHS` y `BATCH_SIZE` si quieres salir más rápido.
- Observa las curvas de pérdida y accuracy para comparar optimizadores.

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
print('TensorFlow', tf.__version__)


TensorFlow 2.19.0


In [2]:
# Cargar Fashion-MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
# añadir canal si necesario
x_train = x_train[..., None]
x_test = x_test[..., None]
print('Train', x_train.shape, y_train.shape)


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Train (60000, 28, 28, 1) (60000,)


In [3]:
def build_model():
    return tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28,28,1)),
        tf.keras.layers.Conv2D(32,3,activation='relu'),
        tf.keras.layers.MaxPool2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])


In [ ]:
EPOCHS = 5
BATCH_SIZE = 128

def run_experiment(optimizer, name):
    model = build_model()
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    history = model.fit(x_train, y_train,
                        validation_split=0.1,
                        epochs=EPOCHS,
                        batch_size=BATCH_SIZE,
                        verbose=2)
    return history


In [ ]:
optimizers = [
    (tf.keras.optimizers.SGD(learning_rate=0.01), 'SGD'),
    (tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9), 'SGD+momentum'),
    (tf.keras.optimizers.RMSprop(learning_rate=0.001), 'RMSProp'),
    (tf.keras.optimizers.Adam(learning_rate=0.001), 'Adam'),
    (tf.keras.optimizers.Adam(learning_rate=0.001, weight_decay=1e-4), 'AdamW_like'),
]

results = {}
for opt, name in optimizers:
    print('Running', name)
    hist = run_experiment(opt, name)
    results[name] = hist

# Plot comparison (loss)
plt.figure(figsize=(12,5))
for name, h in results.items():
    plt.plot(h.history['val_loss'], label=name)
plt.title('Validation loss comparison')
plt.legend()
plt.show()

# Plot comparison (accuracy)
plt.figure(figsize=(12,5))
for name, h in results.items():
    plt.plot(h.history['val_accuracy'], label=name)
plt.title('Validation accuracy comparison')
plt.legend()
plt.show()


## Notas finales y ejercicios
- Prueba con distintos `learning_rate` y `batch_size`.
- Repite los experimentos con `weight_decay` y compara con L2 regularization.
- Extensión: añadir `tf.keras.callbacks.ReduceLROnPlateau` y `LearningRateScheduler` para comparar.